In [20]:
from dotenv import load_dotenv
load_dotenv()


True

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [22]:
loader=PyPDFLoader("../data/data_science_syllabus.pdf")
docs=loader.load()
len(docs)

10

In [23]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data=splitter.split_documents(docs)
len(splitted_data)

10

In [24]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8581.06it/s]


In [25]:
vector_store=Chroma.from_documents(
    documents=splitted_data,
    embedding=embeddings
)

In [26]:
query="Machine learning and data science content"
data= vector_store.similarity_search(query=query)

In [27]:
data[0]

Document(id='1788ae00-12cb-4090-9579-804a922a7be4', metadata={'source': '../data/data_science_syllabus.pdf', 'page': 9, 'total_pages': 10, 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/137.0.0.0 Safari/537.36', 'moddate': '2025-12-13T08:07:26+00:00', 'title': '📘 1-Year Roadmap: Data Analytics, Data Science & GenAI', 'creationdate': '2025-12-13T08:07:26+00:00', 'producer': 'Skia/PDF m137', 'page_label': '10'}, page_content='📘  1\ue088Year Roadmap: Data Analytics, Data Science & GenAI\n10')

In [28]:
context=""
for doc in data:
    context+=doc.page_content + "\n"
print(context)

📘  1Year Roadmap: Data Analytics, Data Science & GenAI
10
📘  1Year Roadmap: Data Analytics, Data Science & GenAI
10
🎤  Mock Interview: Stats Scenarios  Probabilities  Tests
✅  Module 6: Machine Learning – I (Supervised 
Learning) (4 weeks)
Duration: Month 6
Topics:
ML pipeline
Regression: Linear, Logistic
Decision Tree, Random Forest, KNN
Train-test split, model evaluation
Tools:
Scikit-learn, Google Colab, ChatGPT, PyCaret (optional)
Mini Project:
Loan Approval or House Price Prediction
Predict Diabetes from health dataset
🎤  Mock Interview: Supervised Learning Models  Metrics
✅  Module 7: Machine Learning – II (Unsupervised & 
Feature Engineering) (3 weeks)
Duration: Month 7
Topics:
KMeans Clustering
Dimensionality Reduction: PCA
Feature selection, encoding, scaling
Model tuning GridSearchCV
Tools:
Scikit-learn, Seaborn, Colab
Mini Project:
📘  1Year Roadmap: Data Analytics, Data Science & GenAI
5
🎤  Mock Interview: Stats Scenarios  Probabilities  Tests
✅  Module 6: Machine 

In [29]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
# res=llm.invoke(f"""Can you provide me the answer based on the 
#                 provided context for my question, 
#                 context:{context} and question: {query}""")
# print(res.content)

### Chain - Context generate | prompt | llm | Strparser

In [37]:
def get_context(query:str):
    data= vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context+=doc.page_content + "\n"

    return {
        "context":context,
        "question":query
    }
    

In [38]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant and provide answer based on the context for user question.
if you don't know the answer, then you can say that 'I don't know.'

Context: {context}

Question: {question}
""")

In [39]:
rag_chain=get_context | prompt | llm

In [40]:
res=rag_chain.invoke("What is the duration of my course ?")

In [42]:
print(res.content)

Your course is designed to be completed in **one year**.


In [45]:
res=rag_chain.invoke("Who is the content of java DSA course?")

In [46]:
print(res.content)

I’m not sure what specific Java DSA course you’re referring to. If you can provide more details (e.g., the provider or syllabus), I’d be happy to help!
